# Projeto De Machine Learning - 2026.1

## Notebook 05 — XAI e Diagnóstico

### Projeto Kyra Pesquisa

*Maria Beatriz Ribeiro, Juliane Oliveira e Emanuel Gandra*

## 1. Objetivo

Este notebook implementa os itens de **XAI e diagnóstico** exigidos pelo checklist da Aula 10,
e adiciona uma etapa de **taxonomia automática** via Naive Bayes.

**Input:** `data/processed/interviews_chunks_modeling.parquet` + outputs do notebook 03  
**Output:** `outputs/xai/`  &  `src/models/taxonomy_nb.pkl`

---

### Checklist XAI — Kyra (Aula 10)

- [ ] **3.1** Busca por evidência citável — TF-IDF + Cosseno
- [ ] **3.2** Busca por evidência citável — BM25
- [ ] **4.** Top-3 trechos por tópico NMF salvos com `doc_id` e `chunk_id`
- [ ] **5.** Tabela final de evidências por tópico exportada em JSON
- [ ] **6.** Human-in-the-loop documentado
- [ ] **7.** Auditoria de PII — nenhuma informação pessoal nos outputs
- [ ] **8.** Taxonomia automática com MultinomialNB (18 temas) treinada e serializada

| Seção | O que entrega |
|---|---|
| **3.1 TF-IDF + Cosseno** | Função `buscar_cosseno(query, k)` → `doc_id`, `chunk_id`, score |
| **3.2 BM25** | Função `buscar_bm25(query, k)` → mesma interface, melhor recall |
| **4. Top Trechos** | Para cada tópico NMF, 3 chunks de maior ativação com referência citável |
| **5. Evidências JSON** | Um JSON por tópico consolidando label, termos e 3 evidências auditáveis |
| **6. Human-in-the-Loop** | Fluxo de revisão/retroalimentação documentado em JSON |
| **7. PII** | Regex audit — CPF, telefone, e-mail, CEP |
| **8. Taxonomia NB** | 18 temas definidos, MultinomialNB treinado e salvo em `src/models/taxonomy_nb.pkl` |


---
## 2. Setup e Carregamento

In [ ]:
import pandas as pd
import numpy as np
import re
import json
import unicodedata
from pathlib import Path

# ============================================================
# Paths
# ============================================================
try:
    NOTEBOOK_DIR = Path(__file__).resolve().parent
except NameError:
    NOTEBOOK_DIR = Path.cwd()

PROJECT_DIR = NOTEBOOK_DIR.parent if NOTEBOOK_DIR.name.lower() == 'notebooks' else NOTEBOOK_DIR
DATA_DIR    = PROJECT_DIR / 'data'
PROCESSED_DIR = DATA_DIR / 'processed'
OUTPUTS_DIR   = PROJECT_DIR / 'outputs'
XAI_DIR       = OUTPUTS_DIR / 'xai'
XAI_DIR.mkdir(parents=True, exist_ok=True)

# Pasta do run de clusterização mais recente
_cluster_base = OUTPUTS_DIR / 'clusterizacao_insights_v2'
if _cluster_base.exists():
    _runs = sorted(_cluster_base.glob('*'), reverse=True)
    CLUSTER_RUN_DIR = _runs[0] if _runs else OUTPUTS_DIR / 'clusterizacao'
else:
    CLUSTER_RUN_DIR = OUTPUTS_DIR / 'clusterizacao'
print('Run de clusterização:', CLUSTER_RUN_DIR)

# ============================================================
# Base principal de modelagem
# ============================================================
df = pd.read_parquet(PROCESSED_DIR / 'interviews_chunks_modeling.parquet')
print('Chunks de modelagem:', df.shape)

# ============================================================
# Tópicos NMF
# ============================================================
nmf_topics = pd.read_csv(CLUSTER_RUN_DIR / 'tables' / 'nmf_topics.csv')
print('Tópicos NMF:', nmf_topics.shape)
display(nmf_topics[['topic_id', 'top_terms', 'auto_label_short']])

# ============================================================
# Chunks com tópicos NMF atribuídos (output do notebook 03)
# Estrutura real: nmf_topic_id (str), nmf_topic_weight (float), nmf_topic_num (int)
# ============================================================
df_topics = pd.read_parquet(CLUSTER_RUN_DIR / 'chunks_with_clusters_topics.parquet')
print('Chunks com tópicos:', df_topics.shape)
print('Tópicos NMF únicos:', df_topics['nmf_topic_id'].nunique(),
      '| Clusters únicos:', df_topics['cluster_label'].nunique())


---
## 3. Busca por Evidência Citável

Cada insight gerado pelo modelo precisa estar ligado a um **trecho auditável** da transcrição
(`doc_id`, `chunk_id`). Sem isso, a conclusão não passa no critério de XAI da apresentação final.

Implementamos dois motores de busca com a mesma interface de saída:

### 3.1 TF-IDF + Cosseno

Vetor de consulta projetado no mesmo espaço TF-IDF dos chunks. Rápido e interpretável.


In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

# ============================================================
# Índice TF-IDF — mesmos hiperparâmetros do notebook 03
# ============================================================
CORPUS_COL = 'text_for_keywords'

tfidf_vec = TfidfVectorizer(
    sublinear_tf=True,
    min_df=5,
    max_df=0.65,
    ngram_range=(1, 2),
    token_pattern=r"(?u)\b[a-zA-ZÀ-ÿ][a-zA-ZÀ-ÿ]{2,}\b",
)
corpus = df[CORPUS_COL].fillna('').astype(str).tolist()
X_tfidf = tfidf_vec.fit_transform(corpus)
print(f'Matriz TF-IDF: {X_tfidf.shape}  ({X_tfidf.nnz:,} valores não-zero)')


# ============================================================
# Helpers de normalização de query
# ============================================================
def _preprocess_query(q: str) -> str:
    """Normaliza a query da mesma forma que text_for_keywords."""
    s = unicodedata.normalize('NFKD', q.lower())
    s = ''.join(ch for ch in s if not unicodedata.combining(ch))
    s = re.sub(r'[^a-z0-9\s]', ' ', s)
    return re.sub(r'\s+', ' ', s).strip()


# ============================================================
# Busca TF-IDF + Cosseno
# ============================================================
def buscar_cosseno(query: str, k: int = 5) -> pd.DataFrame:
    """Retorna os k chunks mais próximos da query via TF-IDF + cosseno.

    Retorna: doc_id, chunk_id, score_cosseno, trecho (primeiros 400 chars).
    """
    q_vec  = tfidf_vec.transform([_preprocess_query(query)])
    scores = cosine_similarity(q_vec, X_tfidf).ravel()
    top_idx = scores.argsort()[::-1][:k]
    rows = []
    for i in top_idx:
        r = df.iloc[i]
        rows.append({
            'doc_id':        r['doc_id'],
            'chunk_id':      r['chunk_id'],
            'score_cosseno': round(float(scores[i]), 4),
            'trecho':        str(r.get('text_for_embedding', r.get(CORPUS_COL, '')))[:400],
        })
    return pd.DataFrame(rows)


# --- Teste de sanidade ---
_demo_cosseno = buscar_cosseno('embalagem sustentável refil', k=5)
print('Busca cosseno — embalagem sustentável refil')
display(_demo_cosseno[['doc_id', 'chunk_id', 'score_cosseno', 'trecho']])


### 3.2 BM25

BM25 pondera pela frequência do termo e pelo comprimento do documento —
mais preciso que o cosseno puro em chunks de tamanhos variados.
Instale com `pip install rank_bm25` se necessário.

In [ ]:
try:
    from rank_bm25 import BM25Okapi
except ImportError:
    import subprocess, sys
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', 'rank_bm25', '-q'])
    from rank_bm25 import BM25Okapi

# ============================================================
# Índice BM25 — tokenizado sobre text_for_keywords
# ============================================================
_tokenized = [doc.split() for doc in df[CORPUS_COL].fillna('').astype(str)]
bm25_index  = BM25Okapi(_tokenized)
print(f'Índice BM25 construído: {len(_tokenized):,} documentos')


# ============================================================
# Busca BM25
# ============================================================
def buscar_bm25(query: str, k: int = 5) -> pd.DataFrame:
    """Retorna os k chunks mais relevantes via BM25 (Okapi BM25).

    Melhor recall que cosseno em chunks curtos ou vocabulário raro.
    Retorna: doc_id, chunk_id, score_bm25, trecho (primeiros 400 chars).
    """
    tokens = _preprocess_query(query).split()
    scores  = bm25_index.get_scores(tokens)
    top_idx = scores.argsort()[::-1][:k]
    rows = []
    for i in top_idx:
        r = df.iloc[i]
        rows.append({
            'doc_id':     r['doc_id'],
            'chunk_id':   r['chunk_id'],
            'score_bm25': round(float(scores[i]), 4),
            'trecho':     str(r.get('text_for_embedding', r.get(CORPUS_COL, '')))[:400],
        })
    return pd.DataFrame(rows)


# --- Teste de sanidade ---
_demo_bm25 = buscar_bm25('embalagem sustentável refil', k=5)
print('Busca BM25 — embalagem sustentável refil')
display(_demo_bm25[['doc_id', 'chunk_id', 'score_bm25', 'trecho']])

# Comparação rápida: cosseno vs BM25
print('\nCosseno retornou chunk_ids:', _demo_cosseno['chunk_id'].tolist())
print('BM25     retornou chunk_ids:', _demo_bm25['chunk_id'].tolist())


---
## 4. Top Trechos por Tópico NMF

Para cada um dos 18 tópicos NMF, extraímos os **3 chunks mais representativos**
(maior score de ativação) com `doc_id`, `chunk_id` e trecho completo.

Isso é a evidência citável global — vincula cada tópico a textos reais do corpus.

In [ ]:
# ============================================================
# Top-3 chunks por tópico NMF (evidência citável)
# Usa nmf_topic_id + nmf_topic_weight (estrutura real do parquet)
# ============================================================
text_col_chunks = (
    'text_for_embedding' if 'text_for_embedding' in df_topics.columns
    else 'text_for_keywords'
)

top_trechos_rows = []
for _, topic_row in nmf_topics.iterrows():
    t_id      = topic_row['topic_id']        # ex: 'nmf_00'
    t_num     = int(topic_row['topic_num'])  # ex: 0
    label     = topic_row.get('auto_label_short', f'topic_{t_num}')
    top_terms = topic_row.get('top_terms', '')

    top3 = (
        df_topics[df_topics['nmf_topic_id'] == t_id]
        .nlargest(3, 'nmf_topic_weight')
    )

    for rank, (_, row) in enumerate(top3.iterrows(), start=1):
        top_trechos_rows.append({
            'topic_id':  t_num,
            'topic_str': t_id,
            'label':     label,
            'top_terms': top_terms,
            'rank':      rank,
            'doc_id':    row.get('doc_id', ''),
            'chunk_id':  row.get('chunk_id', ''),
            'nmf_score': round(float(row['nmf_topic_weight']), 4),
            'trecho':    str(row.get(text_col_chunks, ''))[:600],
        })

df_top_trechos = pd.DataFrame(top_trechos_rows)
df_top_trechos.to_csv(XAI_DIR / 'top_trechos_nmf.csv', index=False)
print(f'Salvos {len(df_top_trechos)} trechos ({nmf_topics.shape[0]} tópicos × 3) → outputs/xai/top_trechos_nmf.csv')
display(df_top_trechos.head(9))


In [ ]:
# ============================================================
# Exporta evidências agrupadas por tópico (JSON)
# ============================================================
import json as _json

evidencias_payload = []
for tid, grupo in df_top_trechos.groupby('topic_id'):
    evidencias_payload.append({
        'topic_id':   int(tid),
        'label':      grupo['label'].iloc[0],
        'top_terms':  grupo['top_terms'].iloc[0],
        'evidencias': grupo[['rank', 'doc_id', 'chunk_id', 'nmf_score', 'trecho']]
                          .to_dict('records'),
    })

with open(XAI_DIR / 'evidencias_por_topico.json', 'w', encoding='utf-8') as f:
    _json.dump(evidencias_payload, f, ensure_ascii=False, indent=2)

print('Evidências por tópico → outputs/xai/evidencias_por_topico.json')

# Resumo: um tópico por linha
df_resumo = (
    df_top_trechos
    .groupby(['topic_id', 'label'])
    .agg(n_evidencias=('doc_id', 'count'), max_score=('nmf_score', 'max'))
    .reset_index()
)
display(df_resumo)


---
## 5. Human-in-the-Loop

Documentação do fluxo de validação humana: como o analista da Kyra revisa,
corrige e alimenta de volta as predições do modelo.

In [ ]:
# ============================================================
# Human-in-the-Loop — Fluxo de validação/correção
# ============================================================
hitl_flow = {
    "versao": "1.0",
    "data": "2026-05",
    "descricao": (
        "Fluxo de revisão humana dos rótulos gerados pelo modelo NMF e pela "
        "taxonomia Naive Bayes. Analistas da Kyra validam e corrigem os rótulos "
        "antes da entrega final."
    ),
    "etapas": [
        {
            "etapa": 1,
            "acao": "Amostragem estratificada",
            "descricao": "Sortear 10% dos chunks por cluster para revisão manual.",
            "responsavel": "Analista Kyra",
            "input": "chunks_with_clusters_topics.parquet",
            "output": "amostra_revisao.csv",
        },
        {
            "etapa": 2,
            "acao": "Revisão de rótulos NMF",
            "descricao": (
                "Para cada tópico NMF, o analista lê os top-3 trechos e confirma "
                "ou ajusta o rótulo gerado automaticamente."
            ),
            "responsavel": "Analista Kyra",
            "input": "top_trechos_nmf.csv",
            "output": "rotulos_nmf_revisados.csv",
        },
        {
            "etapa": 3,
            "acao": "Revisão da taxonomia NB",
            "descricao": (
                "Para cada cluster, o analista verifica se o tema predito pelo "
                "MultinomialNB condiz com o conteúdo real dos trechos representativos."
            ),
            "responsavel": "Analista Kyra",
            "input": "clusters_taxonomy_nb.csv",
            "output": "clusters_taxonomy_revisados.csv",
        },
        {
            "etapa": 4,
            "acao": "Retroalimentação do modelo",
            "descricao": (
                "Casos corrigidos são adicionados como exemplos de treino para "
                "re-treinar o MultinomialNB na próxima iteração."
            ),
            "responsavel": "Time ML",
            "input": "clusters_taxonomy_revisados.csv",
            "output": "taxonomy_nb_v2.pkl",
        },
    ],
    "criterio_aceitacao": (
        "Concordância analista vs modelo >= 80% por cluster. "
        "Clusters abaixo de 80% são revisados e retroalimentados como exemplos adicionais."
    ),
}

hitl_path = XAI_DIR / 'hitl_fluxo.json'
with open(hitl_path, 'w', encoding='utf-8') as f:
    _json.dump(hitl_flow, f, ensure_ascii=False, indent=2)

print('Fluxo HITL exportado ->', hitl_path)
for etapa in hitl_flow['etapas']:
    print(f"  Etapa {etapa['etapa']}: {etapa['acao']} -> {etapa['output']}")
print('\nCritério de aceitação:', hitl_flow['criterio_aceitacao'])


---
## 6. Privacidade / Auditoria de PII

Verifica se nomes próprios, CPFs, telefones ou e-mails passaram pelo pipeline
e chegaram aos outputs. Garante o requisito de privacidade do produto Kyra.

In [ ]:
# ============================================================
# Auditoria de PII — CPF, telefone, e-mail, CEP
# ============================================================
PII_PATTERNS = {
    'email':    re.compile(r'\b[\w.+-]+@[\w-]+(?:\.[\w-]+)+\b'),
    'telefone': re.compile(r'(?:\+?55\s*)?(?:\(?\d{2}\)?\s*)?\d{4,5}[-\s]?\d{4}'),
    'cpf':      re.compile(r'\b\d{3}\.?\d{3}\.?\d{3}-?\d{2}\b'),
    'cep':      re.compile(r'\b\d{5}-?\d{3}\b'),
}

def auditar_pii(texto: str) -> dict:
    return {tipo: bool(pat.search(str(texto))) for tipo, pat in PII_PATTERNS.items()}

text_col_main = 'text_for_embedding' if 'text_for_embedding' in df.columns else 'text_for_keywords'
pii_flags = df[text_col_main].fillna('').apply(auditar_pii)
df_pii    = pd.DataFrame(pii_flags.tolist())

pii_summary = {
    tipo: {'n_chunks': int(df_pii[tipo].sum()), 'pct': round(df_pii[tipo].mean() * 100, 2)}
    for tipo in PII_PATTERNS
}

print('=== Auditoria de PII — Base de Modelagem ===')
for tipo, stats in pii_summary.items():
    status = 'ATENCAO' if stats['n_chunks'] > 0 else 'OK'
    print(f'  {tipo:12s}: {stats["n_chunks"]:4d} chunks ({stats["pct"]:.2f}%)  [{status}]')

# Relatório completo
df_pii_report = df[['doc_id', 'chunk_id']].copy()
for tipo in PII_PATTERNS:
    df_pii_report[f'pii_{tipo}'] = df_pii[tipo]
df_pii_report['pii_any'] = df_pii.any(axis=1)

df_pii_report.to_csv(XAI_DIR / 'pii_audit.csv', index=False)

flagged = df_pii_report[df_pii_report['pii_any']]
print(f'\nTotal flagged: {len(flagged)} chunks -> outputs/xai/pii_audit.csv')
if len(flagged) > 0:
    display(flagged.head(10))
else:
    print('Nenhum chunk com PII detectado — base limpa.')


---
## 7. Taxonomia Automática com Naive Bayes

Definimos **18 temas** coerentes com o universo de pesquisa da Kyra — embalagem,
sustentabilidade, fragrância, cuidados com a pele, performance de produto, compra,
marca, maternidade, preço, rotina de uso, recomendação, experiência sensorial,
comparação de concorrentes, cabelo, ingredientes, design, sentimento positivo e negativo.

Para cada tema listamos frases-âncora como exemplos de treino. O **MultinomialNB** aprende
a associar o vocabulário ao tema e é aplicado para rotular os 18 clusters existentes.

O modelo treinado é serializado em `src/models/taxonomy_nb.pkl` e pode ser reutilizado
para classificar novos chunks ou novos projetos sem re-treinar.


In [ ]:
# ============================================================
# Taxonomia de 18 temas — documentos de treino por tema
# ============================================================
# Cada entrada é uma lista de "documentos sintéticos" (frases com termos
# característicos do tema). O MultinomialNB usa esses documentos como
# exemplos de treino rotulados.

TAXONOMY_DOCS: dict[str, list[str]] = {
    "embalagem": [
        "embalagem frasco pote tampa lacre vedacao formato compacto",
        "design embalagem plastico vidro papelao descarte apresentacao",
        "refil recarregavel pratico ergonomico leve porta facil",
        "abrir fechar derramar vazar tampa quebrou dificil",
        "embalagem grande pequena cabe bolsa viagem funcional",
        "novo formato redesign embalagem apresentacao visual produto",
    ],
    "sustentabilidade": [
        "refil sustentavel meio ambiente reciclavel descartavel ecologico",
        "biodegradavel residuo lixo reciclagem plastico verde consciente",
        "carbono impacto ambiental reutilizavel recarregar responsavel",
        "eco natureza planeta sustentabilidade consumo consciente escolha",
        "menos plastico embalagem reciclada certificado ambiental",
        "recarregar refil econômico sustentavel menos desperdicio",
    ],
    "fragancia_aroma": [
        "perfume fragrancia cheiro aroma olfativo essencia perfumaria",
        "odor notas floral citrico amadeirado almiscar baunilha",
        "fresco intenso suave duradouro fixacao cheiro agradavel",
        "aroma envolvente seduz marcante leve discreto perfumado",
        "notas de topo fundo coração sillage rastro perfume",
        "cheiro bom diferente marcante suave delicado feminino",
    ],
    "cuidados_pele": [
        "pele hidratante creme locao serum oleo textura absorvido",
        "ressecada oleosa mista sensivel manchas firmeza elasticidade",
        "radiante brilho suavidade maciez tom uniforme pele saudavel",
        "hidratacao diaria rotina pele cuidado corporal facial",
        "vitamina c retinol niacinamida ceramida acido hialuronico pele",
        "pele negra melanina iluminadora uniformizadora cuidado especifico",
    ],
    "performance_produto": [
        "funciona resultado eficacia eficiente duradouro efeito qualidade",
        "resolve melhora perceber diferenca antes depois performance",
        "comprovado entrega promessa expectativa superou surpreendeu",
        "efetivo potente acao rapida visivelmente melhor produto bom",
        "nao funciona decepciona fraco fraqueza resultado esperado",
        "testei resultado longo prazo curto prazo melhora comprovada",
    ],
    "compra_distribuicao": [
        "pedido compra loja online entrega disponivel faltou estoque",
        "prateleira site aplicativo encomenda prazo receber carrinho",
        "frete marketplace revendedor distribuicao comprar onde achar",
        "nao tem disponivel esgotado lancamento nova formula chegou",
        "revendedora consultora catalogo pedido entregue prazo",
        "comprar online loja virtual site oficial entrega rapida",
    ],
    "marca_branding": [
        "marca boticario avon natura nivea kyra confianca reputacao",
        "identidade valores proposta posicionamento imagem legado historia",
        "credibilidade reconhecimento lideranca renome tradicao novidade",
        "marca brasileira nacional importada luxo acessivel premium",
        "confio marca sempre usei fiel leal parceira de anos",
        "marca inovadora moderna classica sofisticada jovem identidade",
    ],
    "maternidade": [
        "bebe filho filha gravidez mae crianca familia amamentacao",
        "gestante infancia maternal cuidado bebe parto recem nascido",
        "bercario maternidade filhos educacao presente para mae",
        "produto bebe linha infantil delicado suave hipoalergenico",
        "grávida gestante produto seguro sem cheiro leve neutro",
        "mae e filho familia cuidado presente rotina bebe",
    ],
    "preco_custo": [
        "preco caro barato custo investimento vale pena economico",
        "acessivel promocao desconto oferta custo beneficio pagar",
        "gasto orcamento valor reais parcelado cashback economizar",
        "caro demais nao compensa barato bom qualidade preco justo",
        "promocao oferta desconto comprou aproveitar liquidacao",
        "custo beneficio otimo vale cada centavo barato qualidade",
    ],
    "rotina_uso": [
        "uso rotina aplico utilizo habito diario semanal frequencia",
        "momento quando usar modo uso instrucao quantidade vez",
        "regularidade incorporar ritual beleza passo a passo sequencia",
        "manha noite antes depois banho aplicar camadas ritual",
        "uso todo dia rotina de cuidado hábito diário",
        "uso regularmente frequência rotina estabelecida incorporei habito",
    ],
    "recomendacao": [
        "recomendo indico amei favorito melhor essencial indispensavel",
        "top nota dez voltaria comprar compraria de novo dica",
        "sugiro divulgo compartilho boca a boca propaganda viral",
        "ja indiquei amigas familia recomendei todo mundo usar",
        "meu preferido favorito nao abro mao indispensavel",
        "recomendo fortemente melhor produto que usei excelente",
    ],
    "experiencia_sensorial": [
        "toque textura consistencia espuma sensacao absorvido pegajoso",
        "leveza pesado cremoso fluido aquoso untoso seco macio",
        "tato pele suave suavidade agradavel sensorial experiencia",
        "textura rica densa leve sensacao pele aplicar espalhar",
        "espuma bastante pouco produto rende absorve rapido demora",
        "sensacao imediata pos aplicacao pele reagiu positivo textural",
    ],
    "comparacao_concorrente": [
        "comparando prefiro melhor diferente alternativa concorrente",
        "versus trocar substituiu testei outros mudei experimentei",
        "trocar marca testar novo voltei para antes troquei",
        "prefiro este ao inves daquele comparei os dois melhor",
        "concorrente lancamento novo versus antigo comparacao direta",
        "substitui por esse melhor que anterior troquei mudança",
    ],
    "cabelo_capilar": [
        "cabelo capilar shampoo condicionador mascara capilar",
        "hidratacao cabelo brilho liso cacheado ondulado crespo",
        "queda crescimento fio couro cabeludo oleosidade coloracao",
        "tintura descoloracao escova progressiva tratamento capilar",
        "cabelo ressecado danificado quimicamente tratamento recuperacao",
        "fio brilhoso hidratado macio anti frizz shampoo bom",
    ],
    "ingredientes_formula": [
        "ingrediente ativo formula componente sulfato parabeno natural",
        "organico quimico principio ativo colageneo acido vitamina",
        "retinol niacinamida ceramida acido hialuronico peptideo antioxidante",
        "sem sulfato sem parabeno clean beauty formula natural",
        "ingredientes naturais organicos veganos cruelty free formula",
        "acido glicólico salicílico azelaico vitamina c e retinol",
    ],
    "design_estetica": [
        "bonito visual cor apresentacao estetica elegante moderno",
        "classico minimalista luxuoso premium sofisticado atrativo chamativo",
        "instagram fotografico vitrine expor design bonito embalagem",
        "lindo produto apresentacao visual impecavel estética incrível",
        "design arrojado diferenciado moderno classico linhas limpas",
        "embalagem bonita cor diferente chama atencao visual lindo",
    ],
    "sentimento_positivo": [
        "amo adoro gosto otimo bom excelente maravilhoso fantastico",
        "incrivel perfeito adorei amei satisfeita feliz felicidade",
        "contente encantada surpresa boa superou aprovo parabens",
        "apaixonada viciada amando usando recomendo top produto",
        "maravilha sensacional excelente nota maxima adorei produto",
        "muito bom surpreendeu positivamente acima expectativas incrivel",
    ],
    "sentimento_negativo": [
        "odeio ruim pessimo horrivel problema reclamei decepciona",
        "decepcionei frustrante insatisfeita nao gostei nao funciona",
        "quebrou vazou estragou irritou alergico reacao arrependida",
        "devolvi reclamei atendimento ruim problema qualidade",
        "nao recomendo nao compro de novo pessima experiencia",
        "frustrante decepcionante nao atende expectativas ruim qualidade",
    ],
}

print(f'Taxonomia definida: {len(TAXONOMY_DOCS)} temas')
for tema, docs in TAXONOMY_DOCS.items():
    print(f'  {tema:30s}: {len(docs)} documentos de treino')


In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.model_selection import cross_val_score
import joblib

# ============================================================
# Construção do corpus de treino sintético
# ============================================================
X_train: list[str] = []
y_train: list[str] = []

for tema, docs in TAXONOMY_DOCS.items():
    for doc in docs:
        X_train.append(_preprocess_query(doc))
        y_train.append(tema)

print(f'Corpus de treino: {len(X_train)} exemplos, {len(set(y_train))} classes')

# ============================================================
# Treino MultinomialNB
# ============================================================
nb_pipeline = Pipeline([
    ('vect', CountVectorizer(
        ngram_range=(1, 2),
        min_df=1,
        token_pattern=r"(?u)\b[a-zA-ZÀ-ÿ][a-zA-ZÀ-ÿ]{1,}\b",
    )),
    ('clf', MultinomialNB(alpha=0.5)),
])

nb_pipeline.fit(X_train, y_train)

cv_scores = cross_val_score(nb_pipeline, X_train, y_train, cv=3, scoring='f1_macro')
print(f'CV F1-macro (treino sintetico, cv=3): {cv_scores.mean():.3f} +/- {cv_scores.std():.3f}')
print(f'Classes ({len(nb_pipeline.classes_)}): {list(nb_pipeline.classes_)}')

# ============================================================
# Serialização
# ============================================================
MODEL_DIR = PROJECT_DIR / 'src' / 'models'
MODEL_DIR.mkdir(parents=True, exist_ok=True)
model_path = MODEL_DIR / 'taxonomy_nb.pkl'
joblib.dump(nb_pipeline, model_path)
print(f'\nModelo salvo -> {model_path}')


In [ ]:
# ============================================================
# Aplicação da taxonomia nos clusters existentes
# cluster_label é o identificador (ex: 'km_013'), cluster_numeric o inteiro
# ============================================================
cluster_col  = 'cluster_label'
text_col_c   = 'text_for_keywords'

cluster_ids = sorted(df_topics[cluster_col].dropna().unique())

def _cluster_text(cid: str) -> str:
    sub = df_topics[df_topics[cluster_col] == cid]
    return ' '.join(sub[text_col_c].fillna('').astype(str).head(5).tolist())

cluster_texts  = [_cluster_text(cid) for cid in cluster_ids]
cluster_norms  = [_preprocess_query(t) for t in cluster_texts]

pred_labels = nb_pipeline.predict(cluster_norms)
pred_probas = nb_pipeline.predict_proba(cluster_norms)
classes     = nb_pipeline.classes_

df_clusters_taxonomy = pd.DataFrame({
    'cluster_label':  cluster_ids,
    'tema_predito':   pred_labels,
    'confianca':      [round(float(p.max()), 3) for p in pred_probas],
    'tema_2o':        [classes[p.argsort()[::-1][1]] for p in pred_probas],
    'confianca_2o':   [round(float(p[p.argsort()[::-1][1]]), 3) for p in pred_probas],
})

# Junta label TF-IDF original (colunas reais: cluster_label, auto_label_short)
_label_path = CLUSTER_RUN_DIR / 'tables' / 'cluster_tfidf_labels.csv'
if _label_path.exists():
    df_tfidf_labels = pd.read_csv(_label_path)
    df_clusters_taxonomy = df_clusters_taxonomy.merge(
        df_tfidf_labels[['cluster_label', 'auto_label_short']].rename(
            columns={'auto_label_short': 'label_tfidf'}
        ),
        on='cluster_label', how='left',
    )

df_clusters_taxonomy.to_csv(XAI_DIR / 'clusters_taxonomy_nb.csv', index=False)
print('Clusters com taxonomia NB -> outputs/xai/clusters_taxonomy_nb.csv')
display(df_clusters_taxonomy)


In [ ]:
# ============================================================
# Validação — distribuição de temas e confiança por cluster
# ============================================================
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

if not df_clusters_taxonomy.empty:
    fig, axes = plt.subplots(1, 2, figsize=(16, 6))

    # Gráfico 1: frequência dos temas preditos
    tema_counts = df_clusters_taxonomy['tema_predito'].value_counts()
    axes[0].barh(tema_counts.index, tema_counts.values, color='steelblue')
    axes[0].set_title('Frequência de Temas Preditos (MultinomialNB)', fontsize=12)
    axes[0].set_xlabel('Nº de Clusters')
    axes[0].invert_yaxis()

    # Gráfico 2: confiança por cluster (verde = alta, vermelho = baixa)
    cores = plt.cm.RdYlGn(df_clusters_taxonomy['confianca'].values)
    labels_y = [
        f"{row.cluster_label} — {row.tema_predito}"
        for row in df_clusters_taxonomy.itertuples()
    ]
    axes[1].barh(labels_y, df_clusters_taxonomy['confianca'].values, color=cores)
    axes[1].axvline(0.5, color='red', linestyle='--', linewidth=1, label='Limiar 0.5')
    axes[1].set_title('Confiança da Predição por Cluster', fontsize=12)
    axes[1].set_xlabel('Confiança')
    axes[1].legend()
    axes[1].invert_yaxis()

    plt.tight_layout()
    fig_path = XAI_DIR / 'taxonomy_nb_clusters.png'
    plt.savefig(fig_path, dpi=120, bbox_inches='tight')
    plt.show()
    print(f'Figura salva -> {fig_path}')

    n_alta = (df_clusters_taxonomy['confianca'] >= 0.5).sum()
    print(f'\nClusters com confianca >= 0.5 : {n_alta}/{len(df_clusters_taxonomy)}')
    print(f'Confianca media               : {df_clusters_taxonomy["confianca"].mean():.3f}')
    print(f'Temas unicos atribuidos       : {df_clusters_taxonomy["tema_predito"].nunique()}/{len(TAXONOMY_DOCS)}')
else:
    print('df_clusters_taxonomy vazio — validacao pulada.')


---
## Checklist Final — Notebook 05

| Item | Status | Saída |
|------|--------|-------|
| 3.1 TF-IDF + Cosseno | Implementado | `buscar_cosseno()` |
| 3.2 BM25 | Implementado | `buscar_bm25()` |
| 4. Top Trechos NMF | Exportado | `outputs/xai/top_trechos_nmf.csv` |
| 5. Evidências por Tópico | Exportado | `outputs/xai/evidencias_por_topico.json` |
| 6. Human-in-the-Loop | Documentado | `outputs/xai/hitl_fluxo.json` |
| 7. Auditoria PII | Executada | `outputs/xai/pii_audit.csv` |
| 8. Taxonomia NB — 18 temas | Treinado + Serializado | `src/models/taxonomy_nb.pkl` |
| 8. Aplicação nos Clusters | 18 clusters rotulados | `outputs/xai/clusters_taxonomy_nb.csv` |
